# 1. Data Pipeline
This notebook builds a Kimball-style data warehouse from Olist Brazilian E-Commerce dataset using an ELT (Extract-Load-Transform) pattern in DuckDB.
## 1.1. Architecture
We will be building 3 separate schemas, each with one responsibility. The purpose of this design is separation of concern.
- **raw** - exact mirror of the source CSVs. No transformation.
- **stage** - processed and clenaed data. Dates parsed, typos fixed, data quality flags added (`is_delivery_complete`, `has_category_name`, `has_dimensions`). Still 1:1 with **raw** but only applied to datasets that will be used (geolocation will not be included)
- **mart** - for analytical purposes. Star schema. Create dimension and fact tables, with surrogate keys and derived measures. Will connect with Power BI here.
## 1.2. Order
1. Load CSVs into `raw` schema
2. Transform `raw` to `stage` with cleaning rules
3. Build `mart` dimensions from `stage`
4. Build `mart` facts from `stage` + `mart` dimensions

# 2. Raw Schema
We will be loading the dataset into a local database, which will be DuckDb. The codeblock below will be creating a `raw` schema to import and load the raw csv files.

In [1]:
import duckdb
from pathlib import Path   

# 1. Connect (creates the field if it doesn't exist)
con = duckdb.connect("../data/olist.duckdb")

# 2. Create the raw schema
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

# 3. Load each CSV into raw.{table_name}
raw_dir = Path("../data/raw")
csv_files = {
    "customers":            "olist_customers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":   "olist_order_payments_dataset.csv",
    "order_reviews":       "olist_order_reviews_dataset.csv",
    "orders":                   "olist_orders_dataset.csv",
    "products":               "olist_products_dataset.csv",
    "sellers":                   "olist_sellers_dataset.csv",
    "translation":            "olist_product_category_name_translation.csv"
}

# loop through the files and load them into duckdb
for table_name, filename in csv_files.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE raw.{table_name} AS 
        SELECT * FROM read_csv('{raw_dir / filename}', header=True, auto_detect=True)
    """)

# 4. Verify
con.execute("SHOW TABLES FROM raw").fetchdf()

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,products
7,sellers
8,translation


In [2]:
for table in csv_files.keys():
    count = con.execute(f"SELECT COUNT(*) FROM raw.{table}").fetchone()[0]
    print(f"{table}: {count:,}")

customers: 99,441
geolocation: 1,000,163
order_items: 112,650
order_payments: 103,886
order_reviews: 99,224
orders: 99,441
products: 32,951
sellers: 3,095
translation: 71


# 3. Stage Schema
First thing we need to is to create the schema for the resulting cleaned datasets. We will be naming it the `stage` schema.

In [3]:
# 1. Create the stage schema
con.execute("CREATE SCHEMA IF NOT EXISTS stage")

## 3.1. Create the tables

In [13]:
# Create stage.customers
con.execute("""
    CREATE OR REPLACE TABLE stage.customers AS
    SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix,
            TRIM(customer_city) AS customer_city,        
            TRIM(customer_state) AS customer_state
    FROM raw.customers
""")

# Create stage.sellers
con.execute("""
    CREATE OR REPLACE TABLE stage.sellers AS
    SELECT
            seller_id,
            seller_zip_code_prefix,
            TRIM(seller_city) AS seller_city,
            TRIM(seller_state) AS seller_state
    FROM raw.sellers
""")

# Create stage.orders
con.execute("""
    CREATE OR REPLACE TABLE stage.orders AS
    SELECT
            order_id,
            customer_id,
            order_status,
            CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
            CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
            CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
            CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
            CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date,
            (order_status = 'delivered'
                AND order_approved_at IS NOT NULL
                AND order_delivered_carrier_date IS NOT NULL
                AND order_delivered_customer_date IS NOT NULL) AS is_delivery_complete
    FROM raw.orders
""")

# Create stage.order_items
con.execute("""
    CREATE OR REPLACE TABLE stage.order_items AS        
    SELECT
            order_id,
            order_item_id,
            product_id,
            seller_id,
            CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
            price,
            freight_value
    FROM raw.order_items
""")

# Create stage.order_reviews
con.execute("""
    CREATE OR REPLACE TABLE stage.order_reviews AS
    SELECT
            review_id,
            order_id,
            review_score,
            review_comment_title,
            review_comment_message,
            CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
            CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
            (review_comment_message IS NOT NULL) AS has_comment
    FROM raw.order_reviews
""")

# Create stage.products
con.execute("""
    CREATE OR REPLACE TABLE stage.products AS
    SELECT
            product_id,
            COALESCE(TRIM(product_category_name), 'Unknown') AS product_category_name,
            product_photos_qty,
            product_name_lenght AS product_name_length,
            product_description_lenght AS product_description_length,
            product_weight_g,
            product_length_cm,
            product_height_cm,
            product_width_cm,
            (product_category_name IS NOT NULL) AS has_category,
            (product_weight_g IS NOT NULL
                AND product_length_cm IS NOT NULL
                AND product_height_cm IS NOT NULL
                AND product_width_cm IS NOT NULL) AS has_dimensions
        FROM raw.products
""")

# Create stage.category_translation
con.execute("""
    CREATE OR REPLACE TABLE stage.category_translation AS
    SELECT
            product_category_name,
            product_category_name_english 
    FROM raw.translation
""")

In [ ]:
# Verify
con.execute("SHOW TABLES FROM stage").fetchdf()

,name
0,category_translation
1,customers
2,order_items
3,order_reviews
4,orders
5,products
6,sellers


## 3.2. Validation

In [16]:
# Check is_delivery_complete is working
con.execute("""
    SELECT 
        order_status,
        COUNT(*) AS total,
        SUM(CASE WHEN is_delivery_complete THEN 1 ELSE 0 END) AS complete
    FROM stage.orders
    GROUP BY order_status
    ORDER BY total DESC
""").fetchdf()

,order_status,total,complete
0,delivered,96478,96455.0
1,shipped,1107,0.0
2,canceled,625,0.0
3,unavailable,609,0.0
4,invoiced,314,0.0
5,processing,301,0.0
6,created,5,0.0
7,approved,2,0.0


In [17]:
# Check Unknown category got applied
con.execute("""
    SELECT COUNT(*) 
    FROM stage.products 
    WHERE product_category_name = 'Unknown'
""").fetchdf()

,count_star()
0,610


In [18]:
# Check has_comment derivation
con.execute("""
    SELECT 
        SUM(CASE WHEN has_comment THEN 1 ELSE 0 END) AS with_comment,
        SUM(CASE WHEN NOT has_comment THEN 1 ELSE 0 END) AS without_comment
    FROM stage.order_reviews
""").fetchdf()

,with_comment,without_comment
0,40977.0,58247.0


# 4. Mart